In [1]:
from pyspark.sql import SparkSession
import pyspark
import re
print(pyspark.__version__)
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
from urllib.parse import urlparse
from pyspark.sql.types import *
from utils import find_new_paths

3.4.1


In [2]:
spark = (
    SparkSession.builder 
    .appName("preprocess_forex") 
    .master("spark://spark-master:7077") 
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate()
    )

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/16 22:30:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/16 22:30:12 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [6]:
df = spark.read.parquet("hdfs://namenode:8020/nifi/forex/forex_2025-11-24_03-30-01.parquet")
df = unpack_results(df)
df.show(5,vertical=True)

-RECORD 0------------------------------
 VolumeTraded      | 1                 
 OpeningPrice      | 3.6715            
 HighestDailyPrice | 3.6715            
 LowestDailyPrice  | 3.6715            
 ClosingPrice      | 3.6715            
 AveragedPrice     | 3.6715            
 NumberOfTrades    | 1                 
 CurrencyTo        | AED               
 Date              | 2025-11-23        
 PartitionDate     | 2025-11-23        
-RECORD 1------------------------------
 VolumeTraded      | 3515              
 OpeningPrice      | 1.546264225630876 
 HighestDailyPrice | 1.548179341094872 
 LowestDailyPrice  | 1.540784567501772 
 ClosingPrice      | 1.547580358110095 
 AveragedPrice     | 1.5472            
 NumberOfTrades    | 3515              
 CurrencyTo        | AUD               
 Date              | 2025-11-23        
 PartitionDate     | 2025-11-23        
-RECORD 2------------------------------
 VolumeTraded      | 1                 
 OpeningPrice      | 81.7629179331307  


In [16]:
df_last = spark.sql("SELECT MAX(DATE) AS latest_date FROM usdexchangerates")
max_date = df_last.collect()[0]['latest_date']
print(max_date)

2025-11-23


In [18]:
df_files = spark.read.parquet("hdfs://namenode:8020/nifi/forex") \
    .withColumn("filename", F.input_file_name())
df_files = df_files.filter(col("file_date") > max_date)

In [9]:
BASE_PATH = "hdfs://namenode:8020/nifi/forex"
META_PATH  = "hdfs://namenode:8020/nifi/metadata/forex_last_timestamp.txt"

def read_last_timestamp(spark, meta_path):
    try:
        return (
            spark.read.text(meta_path)
                 .first()[0]
        )
    except Exception:
        # First run fallback
        return "2025-11-25 00:00:00"

last_ts = read_last_timestamp(spark, META_PATH)
print("Last processed timestamp:", last_ts)

def list_hdfs_files(spark, base_path):
    conf = spark._jsc.hadoopConfiguration()
    uri = spark._jvm.java.net.URI(base_path)
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, conf)
    path = spark._jvm.org.apache.hadoop.fs.Path(base_path)

    return [
        f.getPath().getName()
        for f in fs.listStatus(path)
        if f.isFile() and f.getPath().getName().endswith(".parquet")
    ]

files = list_hdfs_files(spark, BASE_PATH)
print(files)

Last processed timestamp: 2025-11-25 00:00:00
['forex_2025-11-15_14-45-01.parquet', 'forex_2025-11-16_03-30-01.parquet', 'forex_2025-11-17_03-30-00.parquet', 'forex_2025-11-18_03-30-01.parquet', 'forex_2025-11-20_03-30-00.parquet', 'forex_2025-11-21_03-30-01.parquet', 'forex_2025-11-22_03-30-01.parquet', 'forex_2025-11-23_03-30-01.parquet', 'forex_2025-11-23_17-52-18.parquet', 'forex_2025-11-24_03-30-01.parquet', 'forex_2025-11-25_03-30-01.parquet', 'forex_2025-11-26_03-30-01.parquet', 'forex_2025-11-27_03-30-00.parquet', 'forex_2025-11-28_03-30-00.parquet', 'forex_2025-11-29_03-30-00.parquet', 'forex_2025-11-30_03-55-02.parquet', 'forex_2025-12-01_03-30-00.parquet', 'forex_2025-12-02_03-30-01.parquet', 'forex_2025-12-03_03-30-01.parquet', 'forex_2025-12-04_03-30-01.parquet', 'forex_2025-12-05_03-30-01.parquet', 'forex_2025-12-06_03-30-00.parquet', 'forex_2025-12-07_03-30-00.parquet', 'forex_2025-12-08_03-30-01.parquet', 'forex_2025-12-09_03-30-01.parquet', 'forex_2025-12-10_03-30-00.p

In [15]:
def extract_timestamp(fname):
    match = re.search(r"forex_(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})", fname)
    if not match:
        return None
    date_part, time_part = match.groups()
    return f"{date_part} {time_part.replace('-', ':')}"

df_files = spark.createDataFrame(
    [(f, extract_timestamp(f)) for f in files if extract_timestamp(f)],
    ["file_name", "file_ts"]
).withColumn(
    "file_ts", F.to_timestamp("file_ts")
) 
last_ts_lit = F.to_timestamp(F.lit(last_ts))

new_files_df = df_files.filter(F.col("file_ts") > last_ts_lit)

new_files_df.show(30, truncate=False)

new_file_paths = [
    f"{BASE_PATH}/{row.file_name}"
    for row in new_files_df.collect()
]

+---------------------------------+-------------------+
|file_name                        |file_ts            |
+---------------------------------+-------------------+
|forex_2025-11-25_03-30-01.parquet|2025-11-25 03:30:01|
|forex_2025-11-26_03-30-01.parquet|2025-11-26 03:30:01|
|forex_2025-11-27_03-30-00.parquet|2025-11-27 03:30:00|
|forex_2025-11-28_03-30-00.parquet|2025-11-28 03:30:00|
|forex_2025-11-29_03-30-00.parquet|2025-11-29 03:30:00|
|forex_2025-11-30_03-55-02.parquet|2025-11-30 03:55:02|
|forex_2025-12-01_03-30-00.parquet|2025-12-01 03:30:00|
|forex_2025-12-02_03-30-01.parquet|2025-12-02 03:30:01|
|forex_2025-12-03_03-30-01.parquet|2025-12-03 03:30:01|
|forex_2025-12-04_03-30-01.parquet|2025-12-04 03:30:01|
|forex_2025-12-05_03-30-01.parquet|2025-12-05 03:30:01|
|forex_2025-12-06_03-30-00.parquet|2025-12-06 03:30:00|
|forex_2025-12-07_03-30-00.parquet|2025-12-07 03:30:00|
|forex_2025-12-08_03-30-01.parquet|2025-12-08 03:30:01|
|forex_2025-12-09_03-30-01.parquet|2025-12-09 03

In [18]:
if new_file_paths:    
    df = spark.read.parquet(*new_file_paths)
    print("Loaded data")    
else:
    print("No new files to process")
    return

Updated last timestamp: 2025-12-16 03:30:00
Loaded data


In [19]:
df.show(3, vertical=True, truncate=False)

-RECORD 0------------------------------------------------------------------------------------------------------------------
 ticker       | C:USDAED                                                                                                   
 queryCount   | 1                                                                                                          
 resultsCount | 1                                                                                                          
 adjusted     | true                                                                                                       
 results      | [{7, 3.6725, 3.6729, 3.6715, 3.6729, 3.6715, 1764547200000, 7}]                                            
 status       | OK                                                                                                         
 request_id   | 1a37ee2933ed9c4cae52d7673aeae209                                                                           
 count  

In [20]:
def unpack_results(df, results_col='results', ticker_col='ticker'):
    """
    Rozpakowuje kolumnę zagnieżdżonych wyników giełdowych i tworzy z niej 
    kolumny analityczne w DataFrame Spark.

    Parametry:
    ----------
    df : pyspark.sql.DataFrame
        DataFrame zawierający kolumnę z wynikami (np. z API finansowego).
    results_col : str, opcjonalnie
        Nazwa kolumny zawierającej zagnieżdżone wyniki (default 'results').
    ticker_col : str, opcjonalnie
        Nazwa kolumny z symbolami instrumentów finansowych (default 'ticker').

    Zwraca:
    -------
    pyspark.sql.DataFrame
        DataFrame z rozpakowanymi kolumnami: VolumeTraded, OpeningPrice,
        HighestDailyPrice, LowestDailyPrice, ClosingPrice, AveragedPrice,
        NumberOfTrades, Date, PartitionDate, CurrencyTo.
        Niepotrzebne kolumny źródłowe są usunięte.
    """

    # Rozpakowanie pierwszego elementu z listy wyników
    df = df.withColumn("res", F.col(results_col).getItem(0))

    # Tworzenie osobnych kolumn z wartościami giełdowymi
    df = df.withColumn("VolumeTraded", F.col("res.v")) \
           .withColumn("OpeningPrice", F.col("res.o")) \
           .withColumn("HighestDailyPrice", F.col("res.h")) \
           .withColumn("LowestDailyPrice", F.col("res.l")) \
           .withColumn("ClosingPrice", F.col("res.c")) \
           .withColumn("AveragedPrice", F.col("res.vw")) \
           .withColumn("timestamp", F.col("res.t")) \
           .withColumn("NumberOfTrades", F.col("res.n"))

    # Czyszczenie kolumny ticker i wyodrębnienie walut
    df = df.withColumn("ticker_clean", F.regexp_replace(F.col(ticker_col), "^C:", "")) \
           .withColumn("CurrencyFrom", F.col("ticker_clean").substr(1, 3)) \
           .withColumn("CurrencyTo", F.expr("substring(ticker_clean, 4, length(ticker_clean))"))

    # Konwersja timestamp na datę i stworzenie kolumny do partycjonowania
    df = df.withColumn("Date", F.to_date(F.from_unixtime(F.col("timestamp") / 1000))) \
           .withColumn("PartitionDate", F.col("Date"))

    # Usunięcie kolumn pomocniczych i zbędnych
    df = df.drop('results', 'res', 'ticker_clean', 'ticker', 
                 'queryCount', 'resultsCount', 'adjusted', 
                 'status', 'request_id', 'count', 'timestamp', 'CurrencyFrom')

    return df

In [21]:
unpacked = unpack_results(df)
unpacked = unpacked.where(unpacked.PartitionDate.isNotNull()) # Usuwamy nulle
unpacked.show(3,vertical=True, truncate=False)

-RECORD 0------------------------------
 VolumeTraded      | 7                 
 OpeningPrice      | 3.6729            
 HighestDailyPrice | 3.6729            
 LowestDailyPrice  | 3.6715            
 ClosingPrice      | 3.6715            
 AveragedPrice     | 3.6725            
 NumberOfTrades    | 7                 
 CurrencyTo        | AED               
 Date              | 2025-12-01        
 PartitionDate     | 2025-12-01        
-RECORD 1------------------------------
 VolumeTraded      | 225528            
 OpeningPrice      | 1.526298116548124 
 HighestDailyPrice | 1.529589916943268 
 LowestDailyPrice  | 1.5223            
 ClosingPrice      | 1.528561165375033 
 AveragedPrice     | 1.5267            
 NumberOfTrades    | 225528            
 CurrencyTo        | AUD               
 Date              | 2025-12-01        
 PartitionDate     | 2025-12-01        
-RECORD 2------------------------------
 VolumeTraded      | 4935              
 OpeningPrice      | 1445.8759         


In [22]:
(unpacked.write
  .mode("append")
  .format("hive")
  .partitionBy("PartitionDate")
  .saveAsTable("USDExchangeRates"))

print("Saved to the table")

if new_file_paths:
    new_max_ts = (
        new_files_df.agg(F.max("file_ts").alias("max_ts"))
                    .first()["max_ts"]
    )
    spark.createDataFrame(
        [(new_max_ts.strftime("%Y-%m-%d %H:%M:%S"),)],
        ["last_processed_timestamp"]
    ).write.mode("overwrite").text(META_PATH)

    print(f"Updated last timestamp: {new_max_ts}")

25/12/16 22:54:50 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

Saved to the table
Updated last timestamp: 2025-12-16 03:30:00


In [15]:
spark.sql("""
    SELECT * FROM USDExchangeRates
        """).show(5)

+----------+----------+--------------+------------+-----------------+-----------------+-----------------+-----------------+-------------+-------------+
|CurrencyTo|      Date|NumberOfTrades|VolumeTraded|     OpeningPrice|     ClosingPrice| LowestDailyPrice|HighestDailyPrice|AveragedPrice|PartitionDate|
+----------+----------+--------------+------------+-----------------+-----------------+-----------------+-----------------+-------------+-------------+
|       AED|2025-11-17|             6|         6.0|           3.6728|           3.6715|           3.6715|           3.6729|       3.6726|   2025-11-17|
|       AUD|2025-11-17|        213932|    213932.0|1.530690341343946|1.539408866995074|1.529051987767584|1.542709924252943|       1.5354|   2025-11-17|
|       ALL|2025-11-17|             5|         5.0| 81.1614181708525|            83.03|  81.152249895489|            83.03|      82.2807|   2025-11-17|
|       ARS|2025-11-17|          4644|      4644.0|        1409.3161|        1386.9951| 

In [14]:
spark.sql("""
    SELECT DATE, COUNT(*) FROM USDExchangeRates
    GROUP BY DATE
    ORDER BY DATE
        """).show()

+----------+--------+
|      DATE|count(1)|
+----------+--------+
|2025-11-14|     120|
|2025-11-16|     121|
|2025-11-17|     121|
|2025-11-19|     118|
|2025-11-20|     110|
|2025-11-21|     119|
|2025-11-23|     119|
+----------+--------+



In [19]:
spark.stop()